# Final Project
Cole Hansen \
April 28th 2026


## Introduction
For my final project, I'm developing a SPICE simulator for Minecraft's redstone circuitry. I'll be using python to build out a library of blocks to reconstruct circuits in simulation. I'm constantly referencing the [wiki](https://minecraft.wiki/w/Redstone_mechanics) to mimic the behavior of the blocks as closely as possible. I'll then be building out a JSON parser to use this library of blocks to build out the circuit for simulation. Then using a similar discrete event simulation to mimic Minecraft's game tick system for scheduling redstone events.

The minecraft community has always had a strong interest in modding and online tools. As an avid minecraft player myself, I often find myself working with redstone and I wanted to create a tool to help my process. Minecraft doesn't offer many debugging tools, or at least accessible debugging tools for players. My end goal with this project is to create an application that will simulate circuits, and provide an array of tools to help debug circuits that minecraft doesn't supply. 

## Background
Minecraft's redstone has regularly evolved over the last 15 years. I'll be unable to build out everything that minecraft has to offer, so I'll be focusing on logic, memory and signal strength circuits. 


### Redstone Components
Redstone circuits are made of redstone components. These components are blocks that are influenced or influence redstone signals. There are broadly three categories of redstone components: Power components, Transmission components and Redstone mechanisms. Power components are used to generate redstone signals. These include the basics like levers and buttons and will turn on other redstone components or generate power. Transmission components are responsible for the propagation of a redstone signal. Lastly, redstone mechanisms, is a catch-all group that contains all the blocks that can be influenced or activated by a redstone signal. 


### Redstone Signal

Redstone signal is analogous to a high-low signal of a wire. Whenever a signal is generated it is propagated along one of three transmission components. A redstone signal has an attribute called signal strength. Most power components create a signal of strength fifteen. Each of the three redstone transmission components will propagate the signal differently. The most common, redstone dust, will decay signal strength at a rate of 1 per block away from the source. 

Besides powering redstone mechanisms, redstone signals can power blocks. Almost every block in the game can be categorized into two groups depending on how they interact with redstone. A transparent block, or a non-conductive block, is a block that cannot be powered by a redstone signal. Transparent blocks, like glass and half-slabs, will not propagate signal into adjacent redstone components. Conductive blocks be powered in two manners: strongly powered and weakly powered. Strongly powered blocks can power adjacent redstone dust, whereas weakly powered blocks cannot. All powered blocks can indirectly power adjacent redstone mechanisms. 

### Game Ticks
The last important topic to cover, game ticks. Minecraft uses a 20Hz clock to simulate discrete events, like redstone circuitry. For redstone specifically, one clock cycle is dedicated to rising edge events, where redstone components receive their inputs, and the next clock cycle where redstone components send their outputs. Colloquially, a redstone tick is two ticks or two cycles of the Minecraft game clock. It is also important to note that the input and output clock cycles are synchronized, where output ticks occur on odd numbered game ticks, and input ticks on even game ticks. 

## Apporach

### Code Organization and Acrhitecture
The code is broken up into three main sections, the first is a block library. This library of blocks is used to recreate minecraft circuitry. The second section is the circuit, this circuit is responsible for connecting all of the components, scheduling inputs and outputs, and running updates. The core model of this system is the circuit section. Lastly, I have the visualization with vpython. The visualizer is quite rudimentary, as I wanted to focus on the core model behind the visualization. In the visualization, I have basic colors associated with each of the blocks, along with specific changes to show a change in state for that block. These sections come together to form the basis of the project. Shown below is an influence diagram showing how the three sections interract to produce the final projecy. 

![Influence Diagram](./media/Influence.png)

The influence diagram above shows the main program flow of the simulation. The block library is used to define the behavior of each of the simulate-able blocks. The circuit is then used to compose these blocks into a class which handles the model. As previously discussed, Minecraft uses a game clock with a tick dedicated to inputs and outputs. I have a clock in the circuit class upon an output tick, will collect all of the outputs for a given game tick, add them to a queue and organize them into a `SignalDict` for each block. Then on the input tick, all of the update functions are ran using the `SignalDicts` collected from the previous tick. These combine to create the redstone tick through which Minecraft redstone runs off of. The VPython simulation also uses these ticks to update the visual aspects of the simulation. The simulator, which is responsible for calling the step function for the model, collects all the block states from the blocks in the circuit class to visually represent in the simulation. 



#### Block Library

To keep this jupyter notebook from becoming unruly, I've kept all of most of my code definitions in separate files to be imported. This was mostly for the block library which is imported below. But it would still be irresponsible for me not to explain how each component works. I selected 10 blocks to represent. These blocks create numerous circuits with several practical examples. I split up the components, just like the aforementioned [wiki](https://minecraft.wiki/w/Redstone_components). 

In the mechanism library I have three blocks, a `RedstoneLamp`, `CopperBulb` and `NoteBlock`. These mechanisms highlight the three responses from redstone signals that we should expect. The `RedstoneLamp` gets turned on whenever a signal powers the lamp. It simply reflects the signal. The `CopperBulb` changes state whenever a signal is applied. Unlike the redstone lamp, the `CopperBulb` keeps it's state whenever the signal is removed. In modern minecraft, the `CopperBulb` is a new key component in memory circuits. Lastly is the `NoteBlock`, which shows off an impulse response. It plays a note on the tick it is powered. If it is powered for several ticks, it only plays a note on the first tick. 

In the power library, I have four components `Redstone Torch`, `Button`, `Lever`, and `RedstoneBlock`. The `RedstoneBlock` is the easiest component to explain. It is a redstone signal source that is constantly on and provides a weak signal to any connected transmission or mechanisms. The `RedstoneTorch` is a more complex constant power source. The `RedstoneTorch` applies a strong signal in the direction it faces, and a weak signal to components that it is not facing. The `RedstoneTorch` can also be turned 'off', as when the block it is on is powered, the torch stop giving outputs. The `Button` and `Lever` components are user activated and provide either a signal for 5 or 10 ticks, or a togglable signal respectively. 

Lastly, the transmission library constitutes the componenets that transmit redstone signal across distances. These include `RedstoneDust` which transmits power instantly across distances, but has a signal loss of 1 for every block away from the source. The `Repeater` resets signal strength to (Strong, 15), as long as there is power applied to the input side. The repeater adds delay to the signal, and can also be used to extend the signal if the original signal was short enough. Lastly, the `Comparator` is used to maintain or manipulate redstone signals. It will either maintain the signal strength from the input side, compare the input to the signals to the left and right of the comparator, or subtract the signals. These components have more complicated behaviors which are reflected in the code, and if you want to learn more, you can check out the Wiki page for the given component found in the table below. 

| Component  | Wiki Page  |
|---|---|
| `RedstoneLamp` | [Wiki](https://minecraft.wiki/w/Redstone_Lamp) |
| `CopperBulb` | [Wiki](https://minecraft.wiki/w/Copper_Bulb) |
| `NoteBlock` | [Wiki](https://minecraft.wiki/w/Note_Block) |
| `RedstoneBlock`  | [Wiki](https://minecraft.wiki/w/Block_of_Redstone)  |
| `RedstoneTorch`  | [Wiki](https://minecraft.wiki/w/Redstone_Torch) |
| `Button` | [Wiki](https://minecraft.wiki/w/Button)  |
| `Lever`  | [Wiki](https://minecraft.wiki/w/Lever)  |
| `RedstoneDust` | [Wiki](https://minecraft.wiki/w/Redstone_Dust) |
| `Repeater` | [Wiki](https://minecraft.wiki/w/Redstone_Repeater) |
| `Comparator` | [Wiki](https://minecraft.wiki/w/Comparator) |




In [1]:
#Block Library
from lib.mechanism import *
from lib.power import *
from lib.transmission import *

### Simulation Design

For each block, or node, in the circuit the following structure applies. 

![SimNode](./media/SimulatioNode.png)

For each node discussed in the influence diagram the specific structure looks like this. Since this is a nodal discrete event simulation, the nodes all experience the same event structure. A component receives an input. I've structure the input such that it has signal strength and a signal type. The the wiki describes two types of signals, I use six. Minecraft updates all blocks one or two taxi-cab blocks away, whereas I only update adjacent blocks. There are some limitations, none of which are within the current scope of the project, namely quasi-connectivity. To get around this update system, I describe more ways that blocks can be powered. In order of the `enum`, 0 is refered to for no power. This is for blocks that need to be updated but cannot power anything. For example, whenever a repeater sends an update to itself, it has a SignalType of 0. The next is 'STRONG', which strongly powers blocks, but is also capable of locking repeaters. SignalType 2, is 'strong' which strongly powers blocks but cannot lock repeaters. Redstone Dust is annoying, it gives off a SignalType 3, 'WEAK', which weakly poweres blocks. This needs to be distinct from other weak signals since adjacent redstone dust is powered simultaneously. Subsequently, SignalType 4 is 'weak' which weakly powers blocks. Our last signal type, SignalType 5, is 'indirect' which powers adjacent mechanisms but is unable to propogate signal without a repeater or comparator. This is mostly used to describe the outputs from a solid block. 

All other behavior mirrors the behavior of the game. The one nice aspect of this simulation is that minecraft is digital and deterministic. The behavior is exact, predictable and consistent. That is to say, there is no configuration, nor parameterization. The only changes to make are in the circuit, not the model. In future work, perhaps the model could be parameterized to the minecraft version, although I would most likely just produce branches for each major minecraft version. 

### Simulation Flow

Initially, the circuit is created from a series of blocks, all with a position, and a series of other initial arguments. These blocks are used to contruct the initial circuit before the simulation starts. The simulation itself, it uses a discrete event simulation where each event is a game tick. The game tick has two halves, where outputs created, they are combined into `SignalDicts` for each block. Then those blocks that have `SignalDict` updates, receive those updates and process them, creating individual outputs. The `SignalDict` is a custom dictionary object which takes a numpy array, pointing to the offset where the input was from as the key, and the signal type and strength as the value. The visualization is responsible for ticking the model. Then, whenever the outputs are sent, the visualization gets all block states and presents them in the notebook. Ticks are currently manual, where every tick must be stepped through.

![SimFlow](./media/Flow.png)

## Verification Process

The verification proccess is remarkably simple for this. I log into minecraft and rebuild the circuit there. Then I'll `/tick freeze`, to pause the game clock, and manually step through the game. I'll include several screenshots from key game ticks. For more complex circuits, I'll use a mod to expose things like signal strength that isn't able to be seen in vanilla, or base, minecraft. 

## Scenarios

I originally had planned to do three scenarios, which I'll outline here. I was only able to code one of these in before submission, the first of the three scenarios. 

### Scenario 1
I want to describe a simple circuit to show off the redstone mechanisms. I'll use a lever, connect to a redstone line to a redstone lamp, note block and bulb. This simulation shows off the redstone dust transmission, the most difficult implementation of the three transmission types. It also shows off all three mechanisms, and how they respond to signals. 

### Scenario 2
Here I want to use repeaters to show off a memory circuit, as shown in figure "Repeater based JK Latch (D)" on the [wiki](https://minecraft.wiki/w/Redstone_circuits/Memory#JK_flip-flops_and_latches). This shows off the logic aspects of redstone, state based tracking, as well redstone torch and repeater behavior. 

### Scenario 3
I want to include one of my own circuits, pictured below. While there is a block that I haven't implemented, I would be able to get around inclusion by manually adjusting where the redstone is facing. The included circuit is a 2bit Demux, which turns one of four lamps based on the 2 bit input. This scenario expands the complexity even more, and includes all three forms of transmission, and half of our overall component library. More importantly, it would serve as a stress test of the system. 

![DeMux](./media/Demux.png)

In [ ]:

from numpy import array

from lib.power import Lever
from lib.transmission import RedstoneDust
from lib.mechanism import RedstoneLamp, NoteBlock, CopperBulb
from core.circuit import Circuit


lever       = Lever(array([0, 0, 0]), facing="up")

dust_1      = RedstoneDust(array([1, 0, 0]), facing=("east", "west"))
dust_2      = RedstoneDust(array([2, 0, 0]), facing=("east", "west"))
dust_3      = RedstoneDust(array([3, 0, 0]), facing=("east", "west"))

lamp        = RedstoneLamp(array([4, 0, 0]))
note_block  = NoteBlock(array([5, 0, 0]))
copper_bulb = CopperBulb(array([4, 1, 0]))

#Assemble Circuit
circuit = Circuit([lever, dust_1, dust_2, dust_3, lamp, note_block, copper_bulb])

print("Initial state (lever OFF)")
circuit.dump()

print("Toggling lever ON")
lever.toggle()
circuit.notify(lever)

# Give the lever's output into the pending queue manually on tick 0 so the
# circuit picks it up.  In a full simulator the UI would fire lever.toggle()
# and then call circuit.step() — the lever's update() returns its output on
# the very next output phase.

for i in range(12):
    phase = "OUTPUT" if circuit.tick % 2 == 0 else "INPUT "
    circuit.step()
    print(f"  game tick {circuit.tick - 1:02d} [{phase}]  "
          f"lamp={'lit  ' if lamp.lit else 'dark '}  "
          f"bulb={'lit  ' if copper_bulb.lit else 'dark '}  "
          f"dust_1.strength={dust_1.strength}  "
          f"dust_3.strength={dust_3.strength}")



print("Toggling lever OFF")
lever.toggle()
circuit.notify(lever)

for i in range(12):
    phase = "OUTPUT" if circuit.tick % 2 == 0 else "INPUT "
    circuit.step()
    print(f"  game tick {circuit.tick - 1:02d} [{phase}]  "
          f"lamp={'lit  ' if lamp.lit else 'dark '}  "
          f"bulb={'lit  ' if copper_bulb.lit else 'dark '}  "
          f"dust_1.strength={dust_1.strength}  "
          f"dust_3.strength={dust_3.strength}")


print("Brief pulse to CopperBulb (toggle test)")
# Point a second lever at the bulb from below
pulse_lever = Lever(array([4, -1, 0]), facing="up")
circuit.register(pulse_lever)
pulse_lever.toggle()   #ON
circuit.notify(pulse_lever)

for i in range(6):
    circuit.step()

print(f"  CopperBulb after pulse ON  --> lit={copper_bulb.lit}")

pulse_lever.toggle()   #OFF
circuit.notify(pulse_lever)
for i in range(6):
    circuit.step()

print(f"  CopperBulb after pulse OFF --> lit={copper_bulb.lit}")

pulse_lever.toggle()
circuit.notify(pulse_lever)
for i in range(6):
    circuit.step()

pulse_lever.toggle()
circuit.notify(pulse_lever)
for i in range(6):
    circuit.step()

print(f"  CopperBulb after 2nd pulse → lit={copper_bulb.lit}  (toggled back!)")

print(f"Final state  (tick {circuit.tick})")
circuit.dump()


Initial state (lever OFF)
=== Circuit snapshot  tick=0 ===
  (0, 0, 0): Lever at [0 0 0]
  (1, 0, 0): Redstone Dust at [1 0 0] with signal strength 0
  (2, 0, 0): Redstone Dust at [2 0 0] with signal strength 0
  (3, 0, 0): Redstone Dust at [3 0 0] with signal strength 0
  (4, 0, 0): RedstoneLamp(unlit) at [4 0 0]
  (4, 1, 0): CopperBulb(dark, unpowered) at [4 1 0]
  (5, 0, 0): NoteBlock(idle) at [5 0 0]

Toggling lever ON
  game tick 00 [OUTPUT]  lamp=dark   bulb=dark   dust_1.strength=0  dust_3.strength=0
  game tick 01 [INPUT ]  lamp=dark   bulb=dark   dust_1.strength=15  dust_3.strength=0
  game tick 02 [OUTPUT]  lamp=dark   bulb=dark   dust_1.strength=15  dust_3.strength=0
  game tick 03 [INPUT ]  lamp=dark   bulb=dark   dust_1.strength=15  dust_3.strength=13
  game tick 04 [OUTPUT]  lamp=lit    bulb=dark   dust_1.strength=15  dust_3.strength=13
[NoteBlock @ [5 0 0]] *PLING*
  game tick 05 [INPUT ]  lamp=lit    bulb=lit    dust_1.strength=15  dust_3.strength=15
  game tick 06 [OUT

## Results
Shown above are the results from scenario one. I've included my own screenshots from the digital testing I reproduced in game. 

![Tick0](./media/Tick0.png)
Tick 0: This tick shows the cirucit, the tick before the lever is flipped. 

![Tick1](./media/Tick1.png)
Tick 1: This tick is right after the lever is flipped. The signal transmits instantly, lights up the lamp, plays a note, and lights the bulb.

![Tick12](./media/Tick12.png)
This tick shows the bulb stays on, but the note isn't playing nor is the lamp on. 

### Interpretation
I'm have a bug where the adjacent redstone dust likes to keep itself powered, since there's an adjacent dust that's powered. But after testing individual components, I was happy to see that the mechanisms worked as intended. 

### Scenarios Not Implemented

For the unimplemented scenarios, I would need to take the time to develop them, or to develop a way to import the circuits from a JSON schema like [litematica](https://github.com/maruohon/litematica). For the memory circuit, I would be testing each major section for the circuits, as well as testing for intended behavior. 

## Evaluation

As it stands, my simulation could use serious improvement. I need to iron out the redstone dust mechanics, and begin stress testing on larger and more complicated circuitry. I would still argue the basic premise was addressed. Given a redstone circuit, I'm (mostly) able to simulate the circuit out of the game to get more information about signal strength, block states, etc. I really enjoyed working on this project, and I wish I had more time to put in towards making this as accurate as possible. 

### Limitations
There are several limitations for this current model, most notably is the lack of available blocks for designing circuits. Even with all the blocks there are still fundamental flaws in this simulation that stop it from being able to implement all redstone circuits. This style model can never implement entity based redstone, whether that's item elevators, mob farms, copper golem sorters, etc. 

### Application
This model performs best at modelling logic and stateful minecraft redstone. People have created computers to graphing calculators using these types of circuits, and I think that debugging these circuits is the best application for this model. 

### Challenges
I ran into so many little hitches during testing. The redstone line should be fixed by checking if redstone has a valid input source for each tick, which is inneficient but most likely necessary to deal with this. For some of the problems that I did fix, I spent a while figuring out how I wanted to schedule delays, especially since each block is responsible for implementing it's own behavior. I first planned to have a priority queue which extended several ticks into the future for my components to schedule updates. This added a lot of complexity to the circuit class, that I figured there must be a better way. I currently like the self referential way of delaying. This way, the component is still in charge of itself, but it will send the proper update when it occurs. I also had issues implementing inputs and outputs. I knew I wanted to use a dictionary with the block position as the key, and signal as the output. But numpy arrays aren't hashable, and the signal strength needed to be encoded. I thought about using normal tuples for the keys, but then I had to do a conversion every time I needed to do matrix math to find positions. Since I always needed to be able to handle position math, either to find offsets, or to check what direction a position would be from the block, the numpy array had to stay. I then gave up, so to say, and created a custom dictionary since I couldn't find a convenient solution to my problem. 

### Alternative Approaches
Because I'm simulating a digital application, I'm not sure if there's a better way to simulate this but there are certainly other ways to approach simulation. I relied more on the discrete event simulation, than a true SPICE simulation. I think a more nodal approach is also a totally valid way of approaching this problem. I might also suggest trying a more continuous event simulation. For entity based redstone, simulating events in real time, especially for events that don't occur on a tick (like mob movement), this would expand the approach to several different applications, at the expense of accuracy. 

## Conclusions

This project used discrete time simulation and nodal architecture to produce a simulation and debugging platform for minecraft redstone circuits. Due to time constraints, I wasn't able to develop a full system. This platform modeled the architecture of minecraft redstone well, but needs significant work to get to a functional point. 

### Future Work
I do plan on continuing to work on this, without VPython, as an open source project. I'll be spending a lot of time ironing out all the kinks and bugs. The next step would be to keep expanding the block library until I cover all of the major components. After that, I'll be working to accept .litematica files, for easier circuitry building. Building from individual instances is really annoying for larger circuits.